# Historical research notebook

Sanitized source archive, not a validated current model. Saved outputs were removed. Data, credentials and model weights are not included. Path placeholders are not automatically expanded. See docs/CODE_CHANGELOG.md.

**Known leakage case:** same-game batting results enter features in this historical notebook. Do not use its metrics as valid pregame prediction performance. The historical calculation is preserved for review.


# MLB 홈승 예측 모델 v2 - 종결급

### v1 대비 개선
1. 3개년 통합 (학습 데이터 3배)
2. 피처 20개+ (run_margin 통일)
3. Val/Test 완전 분리
4. 3모델 앙상블 + 캘리브레이션
5. 연도별 가중치 + 강한 정규화

## 셀 1: 환경 설정 + 드라이브 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, accuracy_score, brier_score_loss
from sklearn.calibration import calibration_curve
from sklearn.isotonic import IsotonicRegression
import warnings
warnings.filterwarnings('ignore')
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import xgboost as xgb
import lightgbm as lgb
BASE = '<DRIVE_ROOT>/MLB데이터'
print("환경 준비 완료")

## 셀 2: 3개년 데이터 로드

In [ ]:
def load_season(year):
    b = f'{BASE}/{year}'
    results = pd.read_csv(f'{b}/mlb_{year}_정규시즌_경기결과.csv')
    results['날짜'] = pd.to_datetime(results['날짜'])
    results['홈승'] = (results['홈팀결과'] == '승').astype(int)
    results['시즌'] = year
    pit_game = pd.read_csv(f'{b}/mlb_{year}_정규시즌_투수개인기록.csv')
    bat_game = pd.read_csv(f'{b}/mlb_{year}_정규시즌_타자개인기록.csv')
    try: sc_p = pd.read_csv(f'{b}/mlb_{year}_statcast_투수.csv')
    except: sc_p = pd.DataFrame()
    try: sc_b = pd.read_csv(f'{b}/mlb_{year}_statcast_타자.csv')
    except: sc_b = pd.DataFrame()
    return results, pit_game, bat_game, sc_p, sc_b

all_results, all_pit, all_bat, all_sc_p, all_sc_b = [], [], [], [], []
for year in [2023, 2024, 2025]:
    print(f"=== {year}시즌 로딩 ===")
    r, p, b, sp, sb = load_season(year)
    print(f"  경기: {len(r)}, 투수: {len(p)}, 타자: {len(b)}")
    all_results.append(r); all_pit.append(p); all_bat.append(b)
    if len(sp) > 0: all_sc_p.append(sp)
    if len(sb) > 0: all_sc_b.append(sb)

results = pd.concat(all_results, ignore_index=True).sort_values('날짜').reset_index(drop=True)
pit_game = pd.concat(all_pit, ignore_index=True)
bat_game = pd.concat(all_bat, ignore_index=True)
sc_pitchers = pd.concat(all_sc_p, ignore_index=True) if all_sc_p else pd.DataFrame()

print(f"\n총 경기: {len(results)}, 시즌별: {results.groupby('시즌').size().to_dict()}")

## 셀 3: 기본 테이블 + 선발투수 식별

In [ ]:
# 선발 = gamePk+팀 기준 이닝 최대 투수
pit_m = pit_game.merge(results[['gamePk','날짜','시즌']], on='gamePk', how='inner')
pit_m['날짜'] = pd.to_datetime(pit_m['날짜'])
starter = pit_m.sort_values('이닝', ascending=False).groupby(['gamePk','팀']).first().reset_index()
starter = starter.sort_values(['선수ID','날짜']).reset_index(drop=True)

# 기본 테이블
df = results[['gamePk','날짜','시즌','홈팀','홈팀ID','원정팀','원정팀ID',
              '홈팀득점','원정팀득점','홈승']].copy()
df['월'] = df['날짜'].dt.month

# 선발투수 ID merge
sp_id = starter[['gamePk','팀','선수ID']].copy()
df = df.merge(sp_id.rename(columns={'선수ID':'홈팀선발ID'}),
              left_on=['gamePk','홈팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
df = df.merge(sp_id.rename(columns={'선수ID':'원정팀선발ID'}),
              left_on=['gamePk','원정팀'], right_on=['gamePk','팀'], how='left').drop(columns='팀')
print(f"기본 테이블: {df.shape}, 선발매칭: {df['홈팀선발ID'].notna().mean():.1%}")

## 셀 4: 투수 피처 (ERA/WHIP/K9/BB9 + xERA)

In [ ]:
# 누적 rolling + 최근5 rolling
for pid, grp in starter.groupby('선수ID'):
    idx = grp.index
    cum_er = grp['자책'].expanding().sum().shift(1)
    cum_ip = grp['이닝'].expanding().sum().shift(1)
    cum_h = grp['피안타'].expanding().sum().shift(1)
    cum_bb = grp['볼넷'].expanding().sum().shift(1)
    cum_k = grp['삼진'].expanding().sum().shift(1)
    starter.loc[idx, 'roll_ERA'] = (cum_er/cum_ip*9).replace([np.inf,-np.inf], np.nan)
    starter.loc[idx, 'roll_WHIP'] = ((cum_h+cum_bb)/cum_ip).replace([np.inf,-np.inf], np.nan)
    starter.loc[idx, 'roll_K9'] = (cum_k/cum_ip*9).replace([np.inf,-np.inf], np.nan)
    starter.loc[idx, 'roll_BB9'] = (cum_bb/cum_ip*9).replace([np.inf,-np.inf], np.nan)
    r5_er = grp['자책'].shift(1).rolling(5, min_periods=2).sum()
    r5_ip = grp['이닝'].shift(1).rolling(5, min_periods=2).sum()
    starter.loc[idx, 'r5_ERA'] = (r5_er/r5_ip*9).replace([np.inf,-np.inf], np.nan)

spf = starter[['gamePk','선수ID','roll_ERA','roll_WHIP','roll_K9','roll_BB9','r5_ERA']].copy()
for side, id_col in [('home','홈팀선발ID'), ('away','원정팀선발ID')]:
    rn = {c: f'{side}_sp_{c.replace("roll_","")}' for c in ['roll_ERA','roll_WHIP','roll_K9','roll_BB9','r5_ERA']}
    rn['선수ID'] = '_sid'
    df = df.merge(spf.rename(columns=rn), left_on=['gamePk', id_col],
                  right_on=['gamePk','_sid'], how='left').drop(columns='_sid', errors='ignore')

# Spread (값 클수록 홈 유리)
df['era_spread'] = df['away_sp_ERA'] - df['home_sp_ERA']
df['whip_spread'] = df['away_sp_WHIP'] - df['home_sp_WHIP']
df['k9_spread'] = df['home_sp_K9'] - df['away_sp_K9']
df['bb9_spread'] = df['away_sp_BB9'] - df['home_sp_BB9']
df['r5era_spread'] = df['away_sp_r5_ERA'] - df['home_sp_r5_ERA']

# xERA (Statcast)
if len(sc_pitchers) > 0:
    xera_map = sc_pitchers.set_index(['player_id','시즌'])['xera'].to_dict()
    df['home_xera'] = df.apply(lambda r: xera_map.get((r['홈팀선발ID'], r['시즌']), np.nan), axis=1)
    df['away_xera'] = df.apply(lambda r: xera_map.get((r['원정팀선발ID'], r['시즌']), np.nan), axis=1)
    df['xera_spread'] = df['away_xera'] - df['home_xera']
df['era_combined'] = df.get('xera_spread', df['era_spread']).fillna(df['era_spread']) if 'xera_spread' in df.columns else df['era_spread']
print(f"투수 피처 완료: {df.shape}")

## 셀 5: 팀 피처 (승률/득실/스트릭)

In [ ]:
team_games = []
for _, row in results.iterrows():
    team_games.append({'gamePk':row['gamePk'],'날짜':row['날짜'],'시즌':row['시즌'],
        '팀':row['홈팀'],'승':int(row['홈팀결과']=='승'),'득점':row['홈팀득점'],'실점':row['원정팀득점']})
    team_games.append({'gamePk':row['gamePk'],'날짜':row['날짜'],'시즌':row['시즌'],
        '팀':row['원정팀'],'승':int(row['원정팀결과']=='승'),'득점':row['원정팀득점'],'실점':row['홈팀득점']})
tg = pd.DataFrame(team_games).sort_values(['팀','날짜']).reset_index(drop=True)

tg['wr10'] = tg.groupby('팀')['승'].transform(lambda x: x.shift(1).rolling(10, min_periods=5).mean())
tg['season_wr'] = tg.groupby(['팀','시즌'])['승'].transform(lambda x: x.expanding().mean().shift(1))
tg['r5_득점'] = tg.groupby('팀')['득점'].transform(lambda x: x.shift(1).rolling(5,min_periods=3).mean())
tg['r5_실점'] = tg.groupby('팀')['실점'].transform(lambda x: x.shift(1).rolling(5,min_periods=3).mean())
tg['rdiff5'] = tg['r5_득점'] - tg['r5_실점']
tg['rdiff10'] = tg.groupby('팀')['득점'].transform(lambda x: x.shift(1).rolling(10,min_periods=5).mean()) - \
    tg.groupby('팀')['실점'].transform(lambda x: x.shift(1).rolling(10,min_periods=5).mean())

def calc_streak(s):
    streak, cur = [], 0
    for v in s:
        cur = max(cur,0)+1 if v==1 else min(cur,0)-1
        streak.append(cur)
    return streak
tg['streak'] = tg.groupby('팀')['승'].transform(lambda x: calc_streak(x.values))
tg['streak_prev'] = tg.groupby('팀')['streak'].shift(1)

tf = tg[['gamePk','팀','wr10','season_wr','rdiff5','streak_prev','rdiff10']].copy()
for pfx, tcol in [('home','홈팀'), ('away','원정팀')]:
    rn = {c: f'{pfx}_{c}' for c in ['wr10','season_wr','rdiff5','streak_prev','rdiff10']}
    df = df.merge(tf.rename(columns={**rn, '팀':'_t'}), left_on=['gamePk',tcol],
                  right_on=['gamePk','_t'], how='left').drop(columns='_t')

df['wr_spread'] = df['home_wr10'] - df['away_wr10']
df['season_wr_spread'] = df['home_season_wr'] - df['away_season_wr']
df['rdiff_spread'] = df['home_rdiff5'] - df['away_rdiff5']
df['streak_spread'] = df['home_streak_prev'] - df['away_streak_prev']
df['rdiff10_spread'] = df['home_rdiff10'] - df['away_rdiff10']
print(f"팀 피처 완료: {df.shape}")

## 셀 6: 타선 OPS + 구장 피처

In [ ]:
# 라인업 OPS
bat_m = bat_game.merge(results[['gamePk','날짜','시즌']], on='gamePk', how='inner')
bat_m['OPS'] = pd.to_numeric(bat_m['OPS'], errors='coerce')
team_ops = bat_m.groupby(['gamePk','팀','홈원정'])['OPS'].mean().reset_index()
for side in ['home','away']:
    ops = team_ops[team_ops['홈원정']==side][['gamePk','OPS']].rename(columns={'OPS':f'{side}_lineup_OPS'})
    df = df.merge(ops, on='gamePk', how='left')
df['ops_spread'] = df['home_lineup_OPS'] - df['away_lineup_OPS']

# 구장별 평균 득점
park_rpg = results.groupby(['시즌','홈팀']).apply(
    lambda x: (x['홈팀득점']+x['원정팀득점']).mean()).reset_index(name='park_rpg')
df = df.merge(park_rpg, on=['시즌','홈팀'], how='left')
print(f"타선+구장 피처 완료: {df.shape}")

## 셀 7: 피치데이터 피처 (구속/헛스윙) - 시간 소요

In [ ]:
sp_pitch_all = []
for year in [2023, 2024, 2025]:
    print(f"  {year} 피치데이터 처리 중...")
    try:
        pitch = pd.read_csv(f'{BASE}/{year}/mlb_{year}_정규시즌_피치데이터.csv',
            usecols=['gamePk','이닝','투수ID','구속','피치결과'])
        pitch = pitch.merge(results[results['시즌']==year][['gamePk','날짜']], on='gamePk')
        pitch['날짜'] = pd.to_datetime(pitch['날짜'])
        first_inn = pitch[pitch['이닝']<=1].groupby(['gamePk','투수ID']).size().reset_index(name='cnt')
        first_inn = first_inn.sort_values('cnt', ascending=False).groupby('gamePk').first().reset_index()
        swstr_vals = {'Swinging Strike','Swinging Strike (Blocked)'}
        pitch['is_swstr'] = pitch['피치결과'].isin(swstr_vals).astype(int)
        sp_p = pitch.merge(first_inn[['gamePk','투수ID']], on=['gamePk','투수ID'])
        sp_game = sp_p.groupby(['gamePk','투수ID']).agg(
            avg_velo=('구속','mean'), swstr_rate=('is_swstr','mean')).reset_index()
        sp_game = sp_game.merge(results[results['시즌']==year][['gamePk','날짜']], on='gamePk')
        sp_game['날짜'] = pd.to_datetime(sp_game['날짜'])
        sp_pitch_all.append(sp_game)
        print(f"    → {len(sp_game)} 경기")
        del pitch
    except Exception as e:
        print(f"    → 실패: {e}")

if sp_pitch_all:
    spd = pd.concat(sp_pitch_all, ignore_index=True).sort_values(['투수ID','날짜'])
    for pid, grp in spd.groupby('투수ID'):
        idx = grp.index
        spd.loc[idx, 'velo_r3'] = grp['avg_velo'].shift(1).rolling(3, min_periods=1).mean()
        spd.loc[idx, 'swstr_r3'] = grp['swstr_rate'].shift(1).rolling(3, min_periods=1).mean()
        season_avg = grp['avg_velo'].expanding().mean().shift(1)
        recent_avg = grp['avg_velo'].shift(1).rolling(3, min_periods=1).mean()
        spd.loc[idx, 'velo_trend'] = recent_avg - season_avg
    spf2 = spd[['gamePk','투수ID','velo_r3','swstr_r3','velo_trend']].copy()
    for side, id_col in [('home','홈팀선발ID'), ('away','원정팀선발ID')]:
        rn = {'velo_r3':f'{side}_velo','swstr_r3':f'{side}_swstr','velo_trend':f'{side}_velo_trend'}
        df = df.merge(spf2.rename(columns=rn), left_on=['gamePk',id_col],
                      right_on=['gamePk','투수ID'], how='left').drop(columns='투수ID', errors='ignore')
    df['velo_spread'] = df['home_velo'] - df['away_velo']
    df['swstr_spread'] = df['home_swstr'] - df['away_swstr']
    del spd
print(f"피치 피처 완료: {df.shape}")

## 셀 8: 피처 선정 (단변량 AUC)

In [ ]:
df = df.drop_duplicates(subset='gamePk', keep='first').reset_index(drop=True)
print(f"최종 테이블: {df.shape}")

candidates = ['era_combined','era_spread','whip_spread','k9_spread','bb9_spread','r5era_spread',
    'wr_spread','season_wr_spread','rdiff_spread','streak_spread','rdiff10_spread',
    'ops_spread','velo_spread','swstr_spread','home_velo_trend','away_velo_trend','park_rpg']
candidates = [f for f in candidates if f in df.columns]

print(f"\n=== 단변량 AUC ({len(candidates)}개) ===\n")
auc_list = []
for col in candidates:
    valid = df[[col,'홈승']].dropna()
    if len(valid) < 100: continue
    auc = roc_auc_score(valid['홈승'], valid[col])
    auc = max(auc, 1-auc)
    auc_list.append({'feature':col, 'AUC':auc, 'miss':df[col].isnull().mean(), 'n':len(valid)})

auc_df = pd.DataFrame(auc_list).sort_values('AUC', ascending=False)
for _, r in auc_df.iterrows():
    m = '★' if r['AUC']>=0.53 else ' '
    print(f"  {m} {r['feature']:25s} AUC={r['AUC']:.4f}  miss={r['miss']:.1%}")

selected_features = auc_df[auc_df['AUC'] >= 0.515]['feature'].tolist()
print(f"\n선택: {len(selected_features)}개 → {selected_features}")

## 셀 9: Train / Val / Test 분리 (핵심!)

In [ ]:
# Train: 2023전체 + 2024전체 + 2025.4-6 | Val: 2025.7 | Test: 2025.8-9
train_mask = (df['시즌'].isin([2023,2024])) | ((df['시즌']==2025) & (df['월']>=4) & (df['월']<=6))
val_mask = (df['시즌']==2025) & (df['월']==7)
test_mask = (df['시즌']==2025) & (df['월']>=8) & (df['월']<=9)

train = df[train_mask].copy()
val = df[val_mask].copy()
test = df[test_mask].copy()
print(f"Train: {len(train)} (2023+2024+2025.4-6)")
print(f"  시즌별: {train.groupby('시즌').size().to_dict()}")
print(f"Val:   {len(val)} (2025.7)")
print(f"Test:  {len(test)} (2025.8-9)")
print(f"홈승률 - Train:{train['홈승'].mean():.4f} Val:{val['홈승'].mean():.4f} Test:{test['홈승'].mean():.4f}")

# 결측 → train median
medians = {}
for col in selected_features:
    medians[col] = train[col].median()
    train[col] = train[col].fillna(medians[col])
    val[col] = val[col].fillna(medians[col])
    test[col] = test[col].fillna(medians[col])

# 연도 가중치
weight_map = {2023: 0.6, 2024: 0.85, 2025: 1.0}
train['w'] = train['시즌'].map(weight_map)
y_train, y_val, y_test = train['홈승'].values, val['홈승'].values, test['홈승'].values
w_train = train['w'].values
print(f"\n가중치: {weight_map}")

## 셀 10: 모델 학습 (LR + XGBoost + LightGBM)

In [ ]:
print("="*60); print("모델 학습"); print("="*60)

# === LR ===
scaler = StandardScaler()
X_tr = scaler.fit_transform(train[selected_features])
X_va = scaler.transform(val[selected_features])
X_te = scaler.transform(test[selected_features])

lr = LogisticRegression(C=0.5, max_iter=2000, random_state=42)
lr.fit(X_tr, y_train, sample_weight=w_train)
lr_tr_p, lr_va_p, lr_te_p = lr.predict_proba(X_tr)[:,1], lr.predict_proba(X_va)[:,1], lr.predict_proba(X_te)[:,1]
print(f"\n[LR] Train:{roc_auc_score(y_train,lr_tr_p):.4f} Val:{roc_auc_score(y_val,lr_va_p):.4f} Test:{roc_auc_score(y_test,lr_te_p):.4f}")
print("계수:", sorted(zip(selected_features, lr.coef_[0]), key=lambda x: abs(x[1]), reverse=True))

# === XGBoost ===
dtrain = xgb.DMatrix(train[selected_features].values, label=y_train, weight=w_train, feature_names=selected_features)
dval = xgb.DMatrix(val[selected_features].values, label=y_val, feature_names=selected_features)
dtest = xgb.DMatrix(test[selected_features].values, label=y_test, feature_names=selected_features)

xg_params = {'objective':'binary:logistic','eval_metric':'auc','max_depth':3,'learning_rate':0.03,
    'subsample':0.7,'colsample_bytree':0.7,'min_child_weight':30,'reg_alpha':2.0,'reg_lambda':8.0,
    'gamma':0.5,'seed':42,'verbosity':0}
xg_model = xgb.train(xg_params, dtrain, 1000, evals=[(dtrain,'train'),(dval,'val')],
    early_stopping_rounds=50, verbose_eval=100)
xg_tr_p, xg_va_p, xg_te_p = xg_model.predict(dtrain), xg_model.predict(dval), xg_model.predict(dtest)
print(f"\n[XGB] best_round={xg_model.best_iteration}")
print(f"  Train:{roc_auc_score(y_train,xg_tr_p):.4f} Val:{roc_auc_score(y_val,xg_va_p):.4f} Test:{roc_auc_score(y_test,xg_te_p):.4f}")
print(f"  Gap: {roc_auc_score(y_train,xg_tr_p)-roc_auc_score(y_test,xg_te_p):.4f}")
imp = xg_model.get_score(importance_type='gain')
print("  Top features:", sorted(imp.items(), key=lambda x:x[1], reverse=True)[:8])

# === LightGBM ===
lgb_tr = lgb.Dataset(train[selected_features], label=y_train, weight=w_train)
lgb_va = lgb.Dataset(val[selected_features], label=y_val, reference=lgb_tr)
lgb_params = {'objective':'binary','metric':'auc','max_depth':3,'learning_rate':0.03,'num_leaves':6,
    'subsample':0.7,'colsample_bytree':0.7,'min_child_samples':40,'reg_alpha':2.0,'reg_lambda':8.0,
    'min_gain_to_split':0.5,'seed':42,'verbose':-1}
lgb_model = lgb.train(lgb_params, lgb_tr, 1000, valid_sets=[lgb_tr,lgb_va],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)])
lgb_tr_p = lgb_model.predict(train[selected_features])
lgb_va_p = lgb_model.predict(val[selected_features])
lgb_te_p = lgb_model.predict(test[selected_features])
print(f"\n[LGB] best_round={lgb_model.best_iteration}")
print(f"  Train:{roc_auc_score(y_train,lgb_tr_p):.4f} Val:{roc_auc_score(y_val,lgb_va_p):.4f} Test:{roc_auc_score(y_test,lgb_te_p):.4f}")

## 셀 11: 앙상블 + 캘리브레이션

In [ ]:
# 단순 평균
ens_te = (lr_te_p + xg_te_p + lgb_te_p) / 3
ens_va = (lr_va_p + xg_va_p + lgb_va_p) / 3

# Val AUC 기반 가중 평균
va = {'LR':roc_auc_score(y_val,lr_va_p),'XGB':roc_auc_score(y_val,xg_va_p),'LGB':roc_auc_score(y_val,lgb_va_p)}
tot = sum(va.values())
w = {k:v/tot for k,v in va.items()}
print(f"가중치: {w}")
wens_va = w['LR']*lr_va_p + w['XGB']*xg_va_p + w['LGB']*lgb_va_p
wens_te = w['LR']*lr_te_p + w['XGB']*xg_te_p + w['LGB']*lgb_te_p

# Isotonic 캘리브레이션 (Val에서 학습)
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(wens_va, y_val)
cal_te = iso.transform(wens_te)

print(f"\n{'Model':25s} {'AUC':>8s} {'Brier':>8s}")
print("-"*45)
for name, prob in [('LR',lr_te_p),('XGB',xg_te_p),('LGB',lgb_te_p),
                    ('Ensemble(avg)',ens_te),('Ensemble(weighted)',wens_te),('Ensemble+Cal',cal_te)]:
    print(f"{name:25s} {roc_auc_score(y_test,prob):8.4f} {brier_score_loss(y_test,prob):8.4f}")

## 셀 12: 확신도 필터 + A안 비교

In [ ]:
A_ACC, A_GAMES = 0.730, 148
best_prob = wens_te  # 최선 모델

te = test.copy()
te['prob'] = best_prob
te['confidence'] = np.abs(te['prob'] - 0.5)
te = te.sort_values('confidence', ascending=False)
te['pred'] = (te['prob'] >= 0.5).astype(int)
te['hit'] = (te['pred'] == te['홈승']).astype(int)

print(f"A안 벤치마크: {A_ACC:.1%} ({A_GAMES}경기)\n")

# Threshold 탐색
print(f"{'Thr':>6s} {'Games':>6s} {'Acc':>8s} {'vs A':>8s}")
print("-"*32)
for thr in np.arange(0.50, 0.72, 0.01):
    mask = (te['prob']>=thr) | (te['prob']<=(1-thr))
    sel = te[mask]
    if len(sel)<20: continue
    acc = sel['hit'].mean()
    m = '★' if acc>=A_ACC else ' '
    print(f"{m} {thr:.2f} {len(sel):>6d} {acc:>7.1%} {acc-A_ACC:>+7.1%}")

# Top N 커브
print(f"\n=== Top N 적중률 ===")
for n in [50,100,A_GAMES,200,300,len(te)]:
    if n>len(te): continue
    acc = te.head(n)['hit'].mean()
    m = '★' if acc>=A_ACC else ' '
    print(f"{m} Top {n:>4d}: {acc:.1%} ({acc-A_ACC:+.1%})")

# 월별
print(f"\n=== 월별 (Top {A_GAMES}) ===")
top = te.head(A_GAMES)
for month in sorted(top['월'].unique()):
    mg = top[top['월']==month]
    print(f"  {month}월: {mg['hit'].mean():.1%} ({len(mg)}경기)")

## 셀 13: 종합 시각화

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('MLB Home Win Prediction v2', fontsize=16, fontweight='bold')

# 1) 캘리브레이션
ax = axes[0,0]
for nm, p in [('LR',lr_te_p),('XGB',xg_te_p),('LGB',lgb_te_p),('Ens',wens_te)]:
    fr, mp = calibration_curve(y_test, p, n_bins=8, strategy='uniform')
    ax.plot(mp, fr, 'o-', label=nm, ms=4)
ax.plot([0,1],[0,1],'k--',alpha=.5); ax.set_title('Calibration'); ax.legend(fontsize=8)

# 2) 확률분포
ax = axes[0,1]
ax.hist(te[te['홈승']==1]['prob'], bins=25, alpha=.6, color='g', label='Home Win', density=True)
ax.hist(te[te['홈승']==0]['prob'], bins=25, alpha=.6, color='r', label='Away Win', density=True)
ax.axvline(.5, color='k', ls='--'); ax.set_title('Prob Distribution'); ax.legend()

# 3) Top N 커브
ax = axes[0,2]
ns = list(range(20,len(te),10))
ax.plot(ns, [te.head(n)['hit'].mean() for n in ns], 'b-', lw=2)
ax.axhline(A_ACC, color='r', ls='--', label=f'A Plan {A_ACC:.0%}')
ax.axhline(test['홈승'].mean(), color='gray', ls=':', label=f'Baseline')
ax.set_xlabel('Games'); ax.set_ylabel('Accuracy'); ax.set_title('Acc by # Games'); ax.legend(fontsize=8)

# 4) Feature Importance
ax = axes[1,0]
imp_s = sorted(xg_model.get_score(importance_type='gain').items(), key=lambda x:x[1], reverse=True)[:12]
ax.barh([x[0] for x in imp_s][::-1], [x[1] for x in imp_s][::-1], color='steelblue')
ax.set_title('XGB Feature Importance')

# 5) 모델비교
ax = axes[1,1]
mods = {'LR':lr_te_p,'XGB':xg_te_p,'LGB':lgb_te_p,'Ens':wens_te,'Cal':cal_te}
aucs = [roc_auc_score(y_test,p) for p in mods.values()]
bars = ax.bar(range(len(mods)), aucs, color=['#3498db','#e67e22','#9b59b6','#2ecc71','#e74c3c'])
ax.set_xticks(range(len(mods))); ax.set_xticklabels(mods.keys())
for i,v in enumerate(aucs): ax.text(i,v+.002,f'{v:.3f}',ha='center',fontsize=9)
ax.set_title('Test AUC')

# 6) 과적합 체크
ax = axes[1,2]
tr_a = [roc_auc_score(y_train,p) for p in [lr_tr_p,xg_tr_p,lgb_tr_p]]
te_a = [roc_auc_score(y_test,p) for p in [lr_te_p,xg_te_p,lgb_te_p]]
x = np.arange(3)
ax.bar(x-.15, tr_a, .3, label='Train', alpha=.7)
ax.bar(x+.15, te_a, .3, label='Test', alpha=.7)
ax.set_xticks(x); ax.set_xticklabels(['LR','XGB','LGB'])
ax.set_title('Overfitting Check'); ax.legend()
for i in range(3): ax.text(i, max(tr_a[i],te_a[i])+.005, f'gap={tr_a[i]-te_a[i]:.3f}', ha='center', fontsize=8)

plt.tight_layout()
plt.savefig(f'{BASE}/mlb_v2_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("저장 완료: mlb_v2_results.png")

## 셀 14: 최종 요약

In [ ]:
best_auc = max(roc_auc_score(y_test,p) for p in [lr_te_p,xg_te_p,lgb_te_p,ens_te,wens_te,cal_te])
top148_acc = te.head(148)['hit'].mean()

print("="*60)
print("  MLB v2 최종 결과")
print("="*60)
print(f"""
[데이터] Train:{len(train)} | Val:{len(val)} | Test:{len(test)} | 피처:{len(selected_features)}개
[Best AUC] {best_auc:.4f}
[Top 148] B안(v2): {top148_acc:.1%} vs A안: {A_ACC:.1%} → {top148_acc-A_ACC:+.1%}

[v1 대비 개선]
  학습데이터: 1,567 → {len(train)} ({len(train)/1567:.1f}배)
  피처: 10 → {len(selected_features)}
  Val 분리: X → O (7월)
  앙상블: X → LR+XGB+LGB
  캘리브레이션: X → Isotonic
""")